In [1]:
import pandas as pd
import numpy as np

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Load dataset
df = pd.read_csv("ML_Final_Sheet.csv")

# 2. Drop non-numeric column (formula is just label)
df = df.drop(columns=["formula"])

# 3. Separate features and target
X = df.drop(columns=["band_gap"])
y = df["band_gap"]

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Feature scaling (important for ML)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 6. Model (Random Forest = best starting point)
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

# 7. Train model
model.fit(X_train, y_train)

# 8. Predictions
y_pred = model.predict(X_test)

# 9. Evaluation
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

# 10. Save model (IMPORTANT for your thesis)
import joblib
joblib.dump(model, "bandgap_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model saved successfully!")

MAE: 0.6248363313137959
R2 Score: 0.6079588616735606
Model saved successfully!


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

# Load data
df = pd.read_csv("ML_Final_Sheet.csv")

# Drop non-numeric
df = df.drop(columns=["formula"])

# Features & target
X = df.drop(columns=["band_gap"])
y = df["band_gap"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# Hyperparameter tuning
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9]
}

grid = GridSearchCV(
    xgb,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

# Train
grid.fit(X_train, y_train)

# Best model
best_model = grid.best_estimator_

# Predict
y_pred = best_model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Best Params:", grid.best_params_)
print("MAE:", mae)
print("R2 Score:", r2)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
1 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\xgboost\core.py", line 729, in inner_f
    return func(**kwargs)
  File "C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\xgboost\sklearn.py", line 1222

Best Params: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 400, 'subsample': 0.9}
MAE: 0.49540914951689485
R2 Score: 0.7012004607302964


In [4]:
df = df[df["band_gap"] > 0.1]

In [5]:
df["chi_diff_AB"] = abs(df["chi_A"] - df["chi_B"])
df["chi_diff_BX"] = abs(df["chi_B"] - df["chi_X"])
df["chi_diff_AX"] = abs(df["chi_A"] - df["chi_X"])

df["r_ratio_AB"] = df["rA"] / df["rB"]
df["r_ratio_BX"] = df["rB"] / df["rX"]

df["log_volume"] = np.log(df["volume"])
df["density_volume"] = df["density"] / df["volume"]

In [6]:
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

def objective(trial):
    model = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 200, 800),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 0, 5),
        reg_lambda=trial.suggest_float("reg_lambda", 0, 5),
        random_state=42
    )

    score = cross_val_score(model, X, y, cv=5, scoring="r2").mean()
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best Params:", study.best_params)
print("Best R2:", study.best_value)

C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-15 16:56:24,766] A new study created in memory with name: no-name-5ca16de0-e1e1-41a5-a50a-78ffa4c2568c
[I 2026-04-15 16:56:25,851] Trial 0 finished with value: 0.49755822505180725 and parameters: {'n_estimators': 228, 'max_depth': 5, 'learning_rate': 0.09584541539884787, 'subsample': 0.7422684353044517, 'colsample_bytree': 0.8348179124451358, 'gamma': 0.57379439719213, 'reg_alpha': 3.2195180661342855, 'reg_lambda': 0.2561447119778931}. Best is trial 0 with value: 0.49755822505180725.
[I 2026-04-15 16:56:26,996] Trial 1 finished with value: 0.5016065487213677 and parameters: {'n_estimators': 784, 'max_depth': 9, 'learning_rate': 0.19481566518218613, 'subsample': 0.9135929146202287, 'colsample_bytree': 0.8469796220623

Best Params: {'n_estimators': 800, 'max_depth': 12, 'learning_rate': 0.054023693772141834, 'subsample': 0.6499411781531893, 'colsample_bytree': 0.7217817122932211, 'gamma': 1.2955553889746205, 'reg_alpha': 2.5362440208540455, 'reg_lambda': 4.387699676134021}
Best R2: 0.5280432778733396


In [7]:
best_params = study.best_params

model = XGBRegressor(**best_params, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Final R2:", r2_score(y_test, y_pred))

Final R2: 0.6629577931091287


In [8]:
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=300, random_state=42)

ensemble = VotingRegressor([
    ('xgb', model),
    ('rf', rf)
])

ensemble.fit(X_train, y_train)

y_pred = ensemble.predict(X_test)

print("Ensemble R2:", r2_score(y_test, y_pred))

Ensemble R2: 0.666088748465388


In [9]:
import joblib
import pandas as pd

# 1. Load your model
model = joblib.load('Perovskite_Regressor_Model.joblib')

# 2. Complete Feature Set (25 columns based on your training data)
columns = [
    'a', 'b', 'c', 'rA', 'rB', 'rX', 'chi_A', 'chi_B', 'chi_X', 'EA_A', 'EA_B', 'EA_X', 
    'tolerance_octahedra', 'volume', 'density', 'chi_diff_BX', 'r_ratio_BX', 
    'lattice_strain', 'packing_index', 'PC1_Size', 'PC2_Shape', 'Stability_Tag', 
    'is_semicor', 'B_site_encoded'
]

# Physical Data for 3 specific test cases:
# 1. CsSnI3: Real-world solar material (~1.3 eV)
# 2. BaSiO3: The Silicate insulator (Actual ~7.3 eV)
# 3. CsZrCl3: The Chloride insulator (Actual ~5.0 eV)
data = [
    [6.21, 6.21, 6.21, 1.88, 1.10, 2.20, 0.79, 1.96, 2.66, 45.5, 107.3, 295.2, 0.92, 239.3, 4.51, 0.70, 0.50, 0.0, 0.65, -2.16, -0.05, 1, 1, 1.0],
    [4.00, 4.00, 4.00, 1.61, 0.40, 1.40, 0.89, 1.90, 3.44, 13.9, 134.1, 141.0, 0.85, 64.0, 4.45, 1.54, 0.28, 0.05, 0.72, -2.66, 0.05, 0, 0, 2.0],
    [5.00, 5.00, 5.00, 1.88, 0.72, 1.81, 0.79, 1.33, 3.16, 45.5, 41.1, 349.0, 0.88, 125.0, 3.82, 1.83, 0.39, 0.03, 0.68, -2.47, 0.01, 1, 1, 3.0]
]

df_test = pd.DataFrame(data, columns=columns)

# 3. Generate Predictions
predictions = model.predict(df_test)

# 4. Compare and Print
materials = ["CsSnI3 (Solar Ref)", "BaSiO3 (Insulator Ref)", "CsZrCl3 (Insulator Ref)"]
actual_gaps = [1.30, 7.30, 5.00]

print("--- THESIS VALIDATION RESULTS ---")
for mat, pred, actual in zip(materials, predictions, actual_gaps):
    error = abs(pred - actual)
    print(f"{mat}: Predicted {pred:.2f} eV | Actual {actual:.2f} eV | Error: {error:.2f} eV")


--- THESIS VALIDATION RESULTS ---
CsSnI3 (Solar Ref): Predicted 2.35 eV | Actual 1.30 eV | Error: 1.05 eV
BaSiO3 (Insulator Ref): Predicted 1.93 eV | Actual 7.30 eV | Error: 5.37 eV
CsZrCl3 (Insulator Ref): Predicted 1.72 eV | Actual 5.00 eV | Error: 3.28 eV


C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


In [10]:
import joblib
import pandas as pd

# 1. Load your model
model = joblib.load('Perovskite_Regressor_Model.joblib')

# 2. Define Features for MAPbI3
# Note: Features like PC1_Size and PC2_Shape are approximated based on lead-iodide systems.
columns = [
    'a', 'b', 'c', 'band_gap', 'rA', 'rB', 'rX', 'chi_A', 'chi_B', 'chi_X', 
    'EA_A', 'EA_B', 'EA_X', 'tolerance_octahedra', 'volume', 'density', 
    'chi_diff_BX', 'r_ratio_BX', 'lattice_strain', 'packing_index', 
    'PC1_Size', 'PC2_Shape', 'Stability_Tag', 'B_site', 'is_semicor', 
    'B_site_encoded'
]

# Physical Data for MAPbI3 (Experimental Eg ~ 1.55 eV)
mapbi3_data = [
    8.86, 8.86, 12.65, 0.0, 2.17, 1.19, 2.20, 2.55, 2.33, 2.66, 
    1.20, 1.10, 3.06, 0.912, 992.0, 4.16, 0.33, 0.54, 0.02, 0.61, 
    -2.10, -0.05, 1, 'Pb', 1, 0.0
]

# Create DataFrame (excluding 'band_gap' if it's the target variable)
# If your model expects 'band_gap' as an input (unlikely), keep it. 
# Usually, we drop the target column before prediction.
df_mapbi3 = pd.DataFrame([mapbi3_data], columns=columns)
if 'band_gap' in df_mapbi3.columns:
    df_mapbi3 = df_mapbi3.drop(columns=['band_gap'])
if 'B_site' in df_mapbi3.columns:
    df_mapbi3 = df_mapbi3.drop(columns=['B_site'])

# 3. Generate Prediction
prediction = model.predict(df_mapbi3)[0]

print("--- MAPbI3 BENCHMARK TEST ---")
print(f"Predicted Bandgap: {prediction:.2f} eV")
print(f"Actual Bandgap:    1.55 eV")
print(f"Error:             {abs(prediction - 1.55):.2f} eV")


--- MAPbI3 BENCHMARK TEST ---
Predicted Bandgap: 2.39 eV
Actual Bandgap:    1.55 eV
Error:             0.84 eV


C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Load data
df = pd.read_csv("ML_Final_Sheet.csv")

# Remove metals (VERY IMPORTANT)
df = df[df["band_gap"] > 0.1]

# Drop formula
df = df.drop(columns=["formula"])

# Feature engineering (IMPORTANT)
df["chi_diff_AB"] = abs(df["chi_A"] - df["chi_B"])
df["chi_diff_BX"] = abs(df["chi_B"] - df["chi_X"])
df["r_ratio_AB"] = df["rA"] / df["rB"]

# Split
X = df.drop(columns=["band_gap"])
y = df["band_gap"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling (MANDATORY for NN)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 🔥 Neural Network Model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(32, activation='relu'),

    Dense(1)  # Output
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse'
)

# Early stopping (VERY IMPORTANT)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Train
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=300,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# Predict
y_pred = model.predict(X_test)

# Metrics
print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 3.0639 - val_loss: 1.6195
Epoch 2/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.6625 - val_loss: 1.4941
Epoch 3/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1.5229 - val_loss: 1.3976
Epoch 4/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.4513 - val_loss: 1.3709
Epoch 5/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1.4089 - val_loss: 1.3934
Epoch 6/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.3454 - val_loss: 1.3063
Epoch 7/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1.3408 - val_loss: 1.2932
Epoch 8/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.2199 - val_loss: 1.2607
Epoch 9/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1.2202 - val_loss: 1.2129
Epoch 10/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1.2033 - val_loss: 1.1890
Epoch 11/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1.1898 - val_loss: 1.1944
Epoch 12/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

from xgboost import XGBRegressor
import optuna

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("ML_Final_Sheet.csv")

# =========================
# 2. CLEAN DATA
# =========================
df = df[df["band_gap"] > 0.1]   # remove metals
df = df.drop(columns=["formula"])

# =========================
# 3. FEATURE ENGINEERING 🔥
# =========================

# Electronegativity differences
df["chi_diff_AB"] = abs(df["chi_A"] - df["chi_B"])
df["chi_diff_BX"] = abs(df["chi_B"] - df["chi_X"])
df["chi_diff_AX"] = abs(df["chi_A"] - df["chi_X"])

# Radius ratios
df["r_ratio_AB"] = df["rA"] / df["rB"]
df["r_ratio_BX"] = df["rB"] / df["rX"]

# Combined physics features
df["avg_chi"] = (df["chi_A"] + df["chi_B"] + df["chi_X"]) / 3
df["chi_std"] = df[["chi_A", "chi_B", "chi_X"]].std(axis=1)

df["log_volume"] = np.log(df["volume"])
df["density_volume"] = df["density"] / df["volume"]

# =========================
# 4. SPLIT
# =========================
X = df.drop(columns=["band_gap"])
y = df["band_gap"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 5. SCALING
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# 6. OPTUNA TUNING 🔥
# =========================
def objective(trial):
    model = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 300, 1000),
        max_depth=trial.suggest_int("max_depth", 4, 12),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        gamma=trial.suggest_float("gamma", 0, 3),
        reg_alpha=trial.suggest_float("reg_alpha", 0, 3),
        reg_lambda=trial.suggest_float("reg_lambda", 1, 5),
        random_state=42
    )

    score = cross_val_score(model, X, y, cv=5, scoring="r2").mean()
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best Params:", study.best_params)
print("Best CV R2:", study.best_value)

# =========================
# 7. FINAL MODEL
# =========================
model = XGBRegressor(**study.best_params, random_state=42)
model.fit(X_train, y_train)

# =========================
# 8. FEATURE SELECTION 🔥
# =========================
importances = model.feature_importances_

important_features = [
    col for col, imp in zip(X.columns, importances) if imp > 0.01
]

print("Selected Features:", important_features)

# Retrain with selected features
X_train_sel = pd.DataFrame(X_train, columns=X.columns)[important_features]
X_test_sel = pd.DataFrame(X_test, columns=X.columns)[important_features]

model.fit(X_train_sel, y_train)

# =========================
# 9. FINAL EVALUATION
# =========================
y_pred = model.predict(X_test_sel)

print("FINAL R2:", r2_score(y_test, y_pred))
print("FINAL MAE:", mean_absolute_error(y_test, y_pred))

[I 2026-04-15 19:35:31,783] A new study created in memory with name: no-name-9fe9f69d-f683-43d0-be03-c498c9344034
[I 2026-04-15 19:35:33,553] Trial 0 finished with value: 0.4867404532707953 and parameters: {'n_estimators': 461, 'max_depth': 4, 'learning_rate': 0.05372867659521286, 'subsample': 0.7216615090569473, 'colsample_bytree': 0.6908094086987148, 'gamma': 2.19707226970021, 'reg_alpha': 1.1549215299018298, 'reg_lambda': 3.7323875695979214}. Best is trial 0 with value: 0.4867404532707953.
[I 2026-04-15 19:35:35,727] Trial 1 finished with value: 0.48713380521559946 and parameters: {'n_estimators': 883, 'max_depth': 8, 'learning_rate': 0.043822203946287754, 'subsample': 0.8156979417338031, 'colsample_bytree': 0.924462446398802, 'gamma': 2.829663538642291, 'reg_alpha': 2.7253465452924797, 'reg_lambda': 2.425694620104111}. Best is trial 1 with value: 0.48713380521559946.
[I 2026-04-15 19:35:37,303] Trial 2 finished with value: 0.5097562560046562 and parameters: {'n_estimators': 460, 'm

Best Params: {'n_estimators': 906, 'max_depth': 7, 'learning_rate': 0.06159710390619356, 'subsample': 0.8495709243960174, 'colsample_bytree': 0.7596558160100622, 'gamma': 0.2569442350166373, 'reg_alpha': 1.6431790329109353, 'reg_lambda': 2.5114198961796985}
Best CV R2: 0.5417131521058047
Selected Features: ['rA', 'rB', 'rX', 'chi_A', 'chi_B', 'chi_X', 'EA_A', 'EA_B', 'EA_X', 'tolerance_factor', 'octahedral_factor', 'a', 'b', 'c', 'volume', 'density', 'chi_diff_AB', 'chi_diff_BX', 'chi_diff_AX', 'r_ratio_AB', 'r_ratio_BX', 'avg_chi', 'chi_std', 'log_volume', 'density_volume']
FINAL R2: 0.7070336755822826
FINAL MAE: 0.5386912021928592
